<a href="https://colab.research.google.com/github/jkeza1/Group5_database-prediction/blob/main/Teen_phone_addiction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📱 Teen Phone Addiction and Lifestyle Survey Analysis
A clean Google Colab notebook to:
- Load the dataset
- Inspect head and tail
- Check for missing values
- Understand schema design ideas


In [28]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split


In [29]:
df = pd.read_csv('https://raw.githubusercontent.com/jkeza1/Group5_database-prediction/refs/heads/main/data/teen_phone_addiction_dataset.csv')

In [30]:
print(df.head())

   ID               Name  Age  Gender          Location School_Grade  \
0   1    Shannon Francis   13  Female        Hansonfort          9th   
1   2    Scott Rodriguez   17  Female      Theodorefort          7th   
2   3        Adrian Knox   13   Other       Lindseystad         11th   
3   4  Brittany Hamilton   18  Female      West Anthony         12th   
4   5       Steven Smith   14   Other  Port Lindsaystad          9th   

   Daily_Usage_Hours  Sleep_Hours  Academic_Performance  Social_Interactions  \
0                4.0          6.1                    78                    5   
1                5.5          6.5                    70                    5   
2                5.8          5.5                    93                    8   
3                3.1          3.9                    78                    8   
4                2.5          6.7                    56                    4   

   ...  Screen_Time_Before_Bed  Phone_Checks_Per_Day  Apps_Used_Daily  \
0  ...       

In [31]:
print(df.tail())

        ID            Name  Age  Gender        Location School_Grade  \
2995  2996     Jesus Yates   16  Female    New Jennifer         12th   
2996  2997  Bethany Murray   13  Female     Richardport          8th   
2997  2998   Norman Hughes   14   Other      Rebeccaton          7th   
2998  2999  Barbara Hinton   17  Female    Ramirezmouth          9th   
2999  3000  Curtis Johnson   17    Male  Lake Alexander         10th   

      Daily_Usage_Hours  Sleep_Hours  Academic_Performance  \
2995                3.9          6.4                    53   
2996                3.6          7.3                    93   
2997                3.2          6.5                    98   
2998                6.7          7.5                    67   
2999                3.5          6.9                    79   

      Social_Interactions  ...  Screen_Time_Before_Bed  Phone_Checks_Per_Day  \
2995                    4  ...                     0.3                    80   
2996                    5  ...    

In [32]:
print(df.isnull().sum())

ID                        0
Name                      0
Age                       0
Gender                    0
Location                  0
School_Grade              0
Daily_Usage_Hours         0
Sleep_Hours               0
Academic_Performance      0
Social_Interactions       0
Exercise_Hours            0
Anxiety_Level             0
Depression_Level          0
Self_Esteem               0
Parental_Control          0
Screen_Time_Before_Bed    0
Phone_Checks_Per_Day      0
Apps_Used_Daily           0
Time_on_Social_Media      0
Time_on_Gaming            0
Time_on_Education         0
Phone_Usage_Purpose       0
Family_Communication      0
Weekend_Usage_Hours       0
Addiction_Level           0
dtype: int64


### Data Processing

In [33]:
# 1. Drop irrelevant columns
df.drop(columns=['ID', 'Name', 'Location'], inplace=True)

# 2. Encode categorical variables
# Ordinal encoding for School_Grade (7th–12th)
grade_order = ['7th', '8th', '9th', '10th', '11th', '12th']
df['School_Grade'] = OrdinalEncoder(categories=[grade_order]).fit_transform(df[['School_Grade']])

# Label encoding for other categorical columns
label_cols = ['Gender', 'Phone_Usage_Purpose', 'Family_Communication']
for col in label_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# 3. Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)  # for numeric columns
for col in label_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# 4. Scale numerical features
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# 5. Feature Engineering
df['Is_Heavy_User'] = (df['Daily_Usage_Hours'] > 5).astype(int)

# 6. Prepare for modeling
target = 'Addiction_Level'
X = df.drop(columns=[target])
y = df[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


/tmp/ipython-input-33-3315525611.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipython-input-33-3315525611.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)',

In [34]:
print(df['Addiction_Level'].dtype)
print(df.columns)

float64
Index(['Age', 'Gender', 'School_Grade', 'Daily_Usage_Hours', 'Sleep_Hours',
       'Academic_Performance', 'Social_Interactions', 'Exercise_Hours',
       'Anxiety_Level', 'Depression_Level', 'Self_Esteem', 'Parental_Control',
       'Screen_Time_Before_Bed', 'Phone_Checks_Per_Day', 'Apps_Used_Daily',
       'Time_on_Social_Media', 'Time_on_Gaming', 'Time_on_Education',
       'Phone_Usage_Purpose', 'Family_Communication', 'Weekend_Usage_Hours',
       'Addiction_Level', 'Is_Heavy_User'],
      dtype='object')


In [35]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

In [36]:
model.fit(X_train, y_train)


RandomForestRegressor(random_state=42)

In [37]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R2): {r2}")

Mean Squared Error (MSE): 0.1291778611080741
R-squared (R2): 0.8672006261935847


In [38]:
joblib.dump(model, "addiction_model_regression.pkl")


['addiction_model_regression.pkl']

# Summary:

**Data Analysis Key Findings**



*   The 'Addiction_Level' column was successfully reverted to its numerical format
(float64) after initially being label encoded.
*   A RandomForestRegressor model was chosen and successfully trained on the prepared data.


*   The model achieved a Mean Squared Error (MSE) of 34.648 and an R-squared (
) score of 0.862 on the test set.
*  The trained regression model was successfully saved as 'addiction_model_regression.pkl'.










**Insights or Next Steps**



*   The R-squared score of 0.862 indicates that the model explains a significant portion of the variance in 'Addiction_Level'. Further analysis could involve hyperparameter tuning to potentially improve performance.
*  Investigate feature importance from the trained RandomForestRegressor to understand which factors contribute most to predicting 'Addiction_Level'.
